In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helgoul packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import StratifiedKFold, train_test_split, KFold, cross_val_score
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
warnings.filterwarnings('ignore')

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv("/kaggle/input/playground-series-s5e7/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s5e7/test.csv")

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
"""
from ydata_profiling import ProfileReport
train_profile = ProfileReport(train, title="Predict People's Character Report")
train_profile
"""

In [ ]:
"""
train_profile.to_file("./titanic_train_data_profile.html")
"""

◆Profile Reportを見た結果<br>
・特徴量同士の相関が高い<br>
・Stage_fearとDrained_after_socializingの構成はかなり似通っている


In [ ]:
# trainとtestを統合する
data = pd.concat([train,test], sort=False)

In [ ]:
data.info()

In [ ]:
print(data.isnull().sum())

In [ ]:
# 分布に基づく欠損値補完関数
def complement_by_distribution(data, column_name):
    """
    欠損値を現在の分布に基づいてランダムに補完する関数
    カテゴリ変数・量的変数のどちらにも対応

    Parameters:
    - data: pd.DataFrame
    - column_name: str(補完対象の列名)

    Returns:
    - data: pd.DataFrame(補完後のデータ)
    """

    # 非欠損値の取得
    non_null = data[column_name].dropna()

    # 出現割合の計算（normalize=Trueで確率になる）
    value_counts = non_null.value_counts(normalize=True).sort_index()

    # 値と確率をリストに変換
    values = value_counts.index.tolist()
    probabilities = value_counts.values.tolist()

    # 欠損数の取得
    num_missing = data[column_name].isnull().sum()

    # ランダムに補完値を生成
    complemented_values = np.random.choice(values, size=num_missing, p=probabilities)

    # 補完の適用
    data.loc[data[column_name].isnull(), column_name] = complemented_values

    return data

Drained_after_socializingとStage_fearを元のYes/Noの分布に基づいてランダムに補完する<br>
⇒レポートで欠損値が10％くらいだったので、そこまで分布が変わることはないだろうと判断<br>

In [ ]:
# カテゴリ変数の補完
for col in ['Drained_after_socializing', 'Stage_fear']:
    data = complement_by_distribution(data, col)

量的変数の補完を行う<br>
分布が安定しているものは、現在の分布に基づいて欠損値を補完しても問題ないと考えた<br>
分布が安定している=正規分布に近いと定義し、歪度が低いものを現在の分布に基づいて補完する<br>
一般に歪度が±1を超えると強い歪みとされ、±5未満ならほぼ対象と見做される<br>

＜歪度が低いと安定していると言える理由＞<br>
・分布が対象
⇒対称な分布では、代表値（平均・中央値）が一致しやすく、統計的推定が安定する<br>
・モデルが学習しやすい
⇒多くの機械学習モデル（特に線形モデルやニューラルネット）は、対称で連続的な分布を前提にしていることが多い

その他の量的変数については、上記で補完した量的変数とDrained_after_socializing、Stage_fearを用いて予測する<br>
profile reportを見た時に全体的に特徴量間の相関が高かったので予測できるのでは？と考えた<br>
歪度が高い特徴量については、代表値やランダムサンプリングで補完すると偏りを強調してしまったり、歪みを悪化させてしまいそう

In [ ]:
# 量的変数の歪度(skewness)をまとめて確認し、歪度が低いものを選定

# 数値型のカラム(今回はfloat型だけ）を抽出
numerical_cols = data.select_dtypes(include=["float64"]).columns

# 歪度の計算
skewness = data[numerical_cols].skew()

#結果の表示
print(skewness)

Friends_circle_sizeとPost_frequencyの欠損値を現在の分布に基づいて補完する

In [ ]:
# 量的変数の補完（歪度が低いもの）
for col in ['Friends_circle_size', 'Post_frequency']:
    data = complement_by_distribution(data, col)

Drained_after_socializing、Stage_fear、Friends_circle_size、Post_frequencyを使って、<br>
Time_spent_Alone 、Social_event_attendance、Going_outsideの欠損値を予測して補完する<br>

補完にはRandomForestRegressorを用いることにした<br>
＜主な理由＞<br>
・非線形関係（複雑なパターン）もとらえることができる<br>
・標準化やスケーリングが不要<br>
・決定木を多数使って平均を取るため過学習のリスクが低い<br>
　⇒補完のような「予測精度よりも安定性が大事」な場面に向いている<br>

In [ ]:
# カテゴリ変数を数値に変換
data['Drained_after_socializing'] = data['Drained_after_socializing'].replace({'Yes': 1, 'No': 0})
data['Stage_fear'] = data['Stage_fear'].replace({'Yes': 1, 'No': 0})
data['Personality'] = data['Personality'].replace({'Extrovert': 1, 'Introvert': 0})

In [ ]:
# 予測に使う特徴量
input_features = ['Drained_after_socializing', 'Stage_fear', 'Friends_circle_size', 'Post_frequency']

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# 欠損値を補完する関数（RandomForestRegressorを使用）
def complement_missing_values(data, target_col, input_cols):
    # 欠損していない行と欠損している行に分ける
    known = data[data[target_col].notnull()]
    missing = data[data[target_col].isnull()]

    # 欠損がある場合のみ補完処理を実行
    if not missing.empty:
        # 学習用データの準備
        X_train_known = known[input_cols]
        y_train_known = known[target_col]

        # モデルの学習
        rf_model = RandomForestRegressor(random_state=42)
        rf_model.fit(X_train_known, y_train_known)

        # 欠損値の予測
        X_missing = missing[input_cols]
        predicted_missing = rf_model.predict(X_missing)

        # 欠損値を補完
        data.loc[data[target_col].isnull(), target_col] = predicted_missing

    return data

In [ ]:
# 対象の変数を順に補完処理を実行
for target in ['Time_spent_Alone', 'Social_event_attendance', 'Going_outside']:
    data = complement_missing_values(data, target, input_features)

In [ ]:
# 補完後の欠損数を確認
missing_summary = data[['Time_spent_Alone', 'Social_event_attendance', 'Going_outside']].isnull().sum()
print("補完後の欠損値数:")
print(missing_summary)

In [ ]:
# 相互作用特徴量の追加
data['Alone_x_Drained'] = data['Time_spent_Alone'] * data['Drained_after_socializing']
data['Alone_x_StageFear'] = data['Time_spent_Alone'] * data['Stage_fear']
data['Social_x_Drained'] = data['Social_event_attendance'] * data['Drained_after_socializing']
data['Social_x_StageFear'] = data['Social_event_attendance'] * data['Stage_fear']
data['Outside_x_Drained'] = data['Going_outside'] * data['Drained_after_socializing']
data['Outside_x_StageFear'] = data['Going_outside'] * data['Stage_fear']


In [ ]:
# これまでtrainとtestを連結してきたdataを分割する
train = data[~data['Personality'].isnull()]
test = data[data['Personality'].isnull()].drop(columns=['Personality'])

In [ ]:
train.info()

In [ ]:
test.info()

In [ ]:
# データセットの作成
X_train = train.drop(columns=['id', 'Personality'])
y_train = train['Personality']
id_train = train['id']

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV
# 手動で設定するハイパーパラメータ
params_lgb = {
    'boosting_type': 'gbdt',
    'objective': 'binary',
    'metric': 'binary_logloss',
    'verbosity': -1,
    'random_state': 42,
    'importance_type': 'gain',
    'n_estimators': 1000,
    'class_weight': 'balanced'
}

# ハイパーパラメータの探索範囲を設定
param_distributions_lgb = {
    'num_leaves': [20, 31, 50],
    'learning_rate': [0.05, 0.1],
    'max_depth': [5, 10]
}


lgb_model = lgb.LGBMClassifier(**params_lgb)


# ランダムサーチを実行
random_search_lgb = RandomizedSearchCV(estimator=lgb_model, 
                                   param_distributions=param_distributions_lgb,
                                   n_iter=10,
                                   scoring='accuracy', 
                                   cv=StratifiedKFold(n_splits=3), 
                                   verbose=-1, 
                                   random_state=42,
                                   n_jobs=-1)

random_search_lgb.fit(X_train, y_train)

# best_paramsとbest_scoreの表示
print("Best parameters found (LightGBM): ", random_search_lgb.best_params_)
print("Best accuracy found (LightGBM): ", random_search_lgb.best_score_)

In [ ]:
import xgboost as xgb
# 手動で設定するハイパーパラメータ
params_xgb = {
    'use_label_encoder': False,
    'eval_metric': 'logloss',
    'random_state': 42,
    'n_estimators': 1000
}

# ハイパーパラメータの探索範囲を設定
param_distributions_xgb = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0]
}

xgb_model = xgb.XGBClassifier(**params_xgb)

# ランダムサーチを実行
random_search_xgb = RandomizedSearchCV(estimator=xgb_model,
                                      param_distributions=param_distributions_xgb,
                                      n_iter=10,
                                      scoring='accuracy',
                                      cv=StratifiedKFold(n_splits=3),
                                      verbose=-1,
                                      random_state=42,
                                      n_jobs=-1
                                     )


random_search_xgb.fit(X_train, y_train)

# best_paramsとbest_scoreの表示
print("Best parameters found (XGBoost)", random_search_xgb.best_params_)
print("Best accuracy found (XGBoost)", random_search_xgb.best_score_)

In [ ]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier

# 手動で設定する基本パラメータ
params_rf = {
    'random_state': 42,
    'n_estimators': 1000
}

# ハイパーパラメータの探索範囲を設定
param_distributions_rf = {
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_model = RandomForestClassifier(**params_rf)

# ランダムサーチを実行
random_search_rf = RandomizedSearchCV(estimator=rf_model,
                                      param_distributions=param_distributions_rf,
                                      scoring='accuracy',
                                      cv=StratifiedKFold(n_splits=3),
                                      verbose=-1,
                                      random_state=42,
                                      n_jobs=-1
                                     )

random_search_rf.fit(X_train, y_train)


# best_paramsとbest_scoreの表示
print("Best parameters found (RandomForest): ", random_search_rf.best_params_)
print("Best accuracy found (RandomForest): ", random_search_rf.best_score_)

In [ ]:
# 最適パラメータの取得とモデル再訓練

# 最適なハイパーパラメータをparamsに追加
best_params_lgb = random_search_lgb.best_params_
params_lgb.update(best_params_lgb)

# 最適なハイパーパラメータでモデルを再訓練
best_model_lgb = lgb.LGBMClassifier(**params_lgb)
best_model_lgb.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(0)
    ]
)

In [ ]:
best_params_xgb = random_search_xgb.best_params_
params_xgb.update(best_params_xgb)
best_xgb_model = xgb.XGBClassifier(**params_xgb)
best_xgb_model.fit(X_train, y_train)

In [ ]:
best_params_rf = random_search_rf.best_params_
params_rf.update(best_params_rf)
best_rf_model = RandomForestClassifier(**params_rf)
best_rf_model.fit(X_train, y_train)

In [ ]:
# 推論用データセットの作成
X_test = test.drop(columns=['id'])
id_test = test['id'] 

In [ ]:
# VotingClassifier の定義（最適化済みモデルを使用）
voting_model = VotingClassifier(
    estimators=[
        ('lgb', best_model_lgb),
        ('xgb', best_xgb_model),
        ('rf', best_rf_model)
    ],
    voting='soft'
)

# 交差検証で評価
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(voting_model, X_train, y_train, cv=cv, scoring='accuracy')
print(f"[VotingClassifier Cross-Validation Results]")
print(f"Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# モデルの学習と予測
voting_model.fit(X_train, y_train)
y_test_pred = voting_model.predict(X_test)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_feature_importance(model, model_name, feature_names):
    if hasattr(model, 'feature_importances_'):
        importance = pd.DataFrame({
            'Feature': feature_names,
            'Importance': model.feature_importances_
        }).sort_values(by='Importance', ascending=False)

        plt.figure(figsize=(10, 6))
        sns.barplot(x='Importance', y='Feature', data=importance, palette='viridis')
        plt.title(f'Feature Importances ({model_name})')
        plt.tight_layout()
        plt.show()

# 各モデルの重要度を表示
plot_feature_importance(best_model_lgb, 'LightGBM', X_train.columns)
plot_feature_importance(best_xgb_model, 'XGBoost', X_train.columns)
plot_feature_importance(best_rf_model, 'RandomForest', X_train.columns)


In [ ]:
# id_testをDataFrameに変換する
id_test_df = pd.DataFrame({'id': id_test.values})  # または id_test.reset_index(drop=True)

# 予測結果をまとめる
submit = pd.DataFrame({'id': id_test_df['id'], 'Personality': pd.Series(y_test_pred).replace({1: 'Extrovert', 0: 'Introvert'})
                      })
display(submit.head())
submit.to_csv('submission_baseline.csv', index=None)